# Stenborg & Cobelli (2003) Wavelet Packets Equalisation — Implementation / 웨이블릿 패킷 평활화 구현

**Paper**: G. Stenborg, P. J. Cobelli. *A wavelet packets equalization technique to reveal the multiple spatial-scale nature of coronal structures*. A&A 398, 1185-1193 (2003).

This notebook reproduces the four-step pipeline of the paper:
1. 2D à trous wavelet transform with $B_3$-spline kernel.
2. Two-level wavelet-packet decomposition (re-decomposing each first-level plane).
3. Local noise variance estimation and hard-thresholding (the $3\sigma$ rule).
4. Weighted recomposition with user-controlled weights $\alpha_{i,j}$.
5. A small experiment comparing different reconstruction strategies on a synthetic coronagraph image.
6. Summary table mapping the paper's concepts to modern equivalents.

Computational scope: 256×256 synthetic coronagraph image, 4-level first decomposition × 3 sub-scales, runs in <30 s on CPU.

본 노트북은 논문의 4단계 파이프라인을 재현한다 — (1) 2D à trous 변환($B_3$-스플라인), (2) 2단계 웨이블릿 패킷 분해, (3) 국소 노이즈 추정 + $3\sigma$ hard-threshold, (4) 가중 재합성. 256×256 합성 영상에서 CPU 30초 이내 실행.

In [ ]:
from typing import List, Sequence, Tuple

import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import convolve

np.random.seed(0)
plt.rcParams['figure.figsize'] = (10, 4)
plt.rcParams['font.size'] = 10

## Part 1: 2D à trous Wavelet Transform / 2D à trous 웨이블릿 변환

The à trous algorithm uses a fixed $B_3$-spline 5×5 separable kernel with values
$\frac{1}{256}\begin{pmatrix}1 & 4 & 6 & 4 & 1\\4 & 16 & 24 & 16 & 4\\6 & 24 & 36 & 24 & 6\\4 & 16 & 24 & 16 & 4\\1 & 4 & 6 & 4 & 1\end{pmatrix}$.
At iteration $i$, we convolve the previous smoothed array with the kernel *dilated by $2^{i-1}$* (the 'à trous' / 'with holes' part), then take the difference to obtain the wavelet plane.

à trous 알고리즘은 $B_3$-스플라인 5×5 분리형 커널을 사용한다. 반복 $i$에서 이전 평활화 영상을 **간격 $2^{i-1}$**로 확장한 커널로 합성곱하고, 차분으로 wavelet plane을 얻는다 — 이것이 'à trous' (구멍 뚫린)의 의미다.

In [ ]:
B3_KERNEL_1D = np.array([1.0, 4.0, 6.0, 4.0, 1.0]) / 16.0
B3_KERNEL_2D = np.outer(B3_KERNEL_1D, B3_KERNEL_1D)  # separable 5x5


def dilate_kernel(kernel: np.ndarray, dilation: int) -> np.ndarray:
    """Insert (dilation - 1) zeros between every pair of kernel taps.

    Args:
        kernel: 2D NumPy array.
        dilation: Dilation factor (>= 1). dilation=1 returns the input.

    Returns:
        Dilated 2D kernel suitable for the à trous step.
    """
    if dilation <= 1:
        return kernel
    h, w = kernel.shape
    H = (h - 1) * dilation + 1
    W = (w - 1) * dilation + 1
    out = np.zeros((H, W), dtype=kernel.dtype)
    out[::dilation, ::dilation] = kernel
    return out


def atrous_decompose(image: np.ndarray, n_scales: int) -> Tuple[List[np.ndarray], np.ndarray]:
    """Apply n-scale à trous wavelet decomposition with B3-spline kernel.

    Args:
        image: 2D float array.
        n_scales: Number of wavelet planes to compute (>= 1).

    Returns:
        Tuple (planes, continuum) where:
            planes is a list of n_scales 2D arrays (same shape as image),
            continuum is the final smoothed array (same shape).
        The exact reconstruction identity is image == continuum + sum(planes).
    """
    planes: List[np.ndarray] = []
    c_prev = image.astype(np.float64)
    for i in range(1, n_scales + 1):
        kernel_i = dilate_kernel(B3_KERNEL_2D, dilation=2 ** (i - 1))
        c_curr = convolve(c_prev, kernel_i, mode='reflect')
        planes.append(c_prev - c_curr)
        c_prev = c_curr
    return planes, c_prev


def atrous_reconstruct(planes: Sequence[np.ndarray], continuum: np.ndarray) -> np.ndarray:
    """Reconstruct image as continuum + sum of planes."""
    return continuum + np.sum(np.stack(planes, axis=0), axis=0)


# Sanity check on a random 64x64 image: reconstruction should be exact.
_test = np.random.randn(64, 64)
_planes, _cont = atrous_decompose(_test, n_scales=4)
_rec = atrous_reconstruct(_planes, _cont)
print(f'Reconstruction error: {np.abs(_rec - _test).max():.2e}')

## Part 2: Two-Level Wavelet Packet Decomposition / 2단계 웨이블릿 패킷 분해

The novelty of the paper: each first-level wavelet plane $w^{(0)}_i$ is *re-decomposed* into sub-scales $\{w^{(0,1)}_i, w^{(0,2)}_i, \ldots\}$ plus its own continuum. We use 3 sub-scales per first-level plane.

본 논문의 차별성: 각 1차 wavelet plane을 다시 분해해 sub-scale 집합을 만든다. 본 구현에서는 3 sub-scale × 4 first-level plane을 사용한다.

In [ ]:
def packet_decompose(
    image: np.ndarray, n_first: int, n_sub: int
) -> Tuple[List[List[np.ndarray]], List[np.ndarray], np.ndarray]:
    """Two-level wavelet packet decomposition.

    Args:
        image: 2D float array.
        n_first: Number of first-level wavelet planes.
        n_sub: Number of sub-scales each first-level plane is re-decomposed into.

    Returns:
        Tuple (subplanes, sub_continua, top_continuum) where:
            subplanes[i][j] is the j-th sub-scale of the i-th first-level plane,
            sub_continua[i] is the continuum of the i-th first-level plane,
            top_continuum is the continuum of the original image.
    """
    first_planes, top_continuum = atrous_decompose(image, n_scales=n_first)
    subplanes: List[List[np.ndarray]] = []
    sub_continua: List[np.ndarray] = []
    for plane in first_planes:
        sps, cs = atrous_decompose(plane, n_scales=n_sub)
        subplanes.append(sps)
        sub_continua.append(cs)
    return subplanes, sub_continua, top_continuum


def packet_reconstruct(
    subplanes: List[List[np.ndarray]],
    sub_continua: List[np.ndarray],
    top_continuum: np.ndarray,
    alpha_top: float,
    alpha_sub_cont: Sequence[float],
    alpha_sub: Sequence[Sequence[float]],
) -> np.ndarray:
    """Weighted recomposition (Eq. 17 of paper).

    Args:
        subplanes: Output of packet_decompose.
        sub_continua: Output of packet_decompose.
        top_continuum: Output of packet_decompose.
        alpha_top: Weight of the top continuum (alpha_{0, 0}).
        alpha_sub_cont: Weights of the sub-continua, length n_first.
        alpha_sub: Weights of sub-planes, shape n_first x n_sub.

    Returns:
        Reconstructed image (same shape as the original).
    """
    out = alpha_top * top_continuum
    for i, (sps, sc, asc) in enumerate(zip(subplanes, sub_continua, alpha_sub_cont)):
        out = out + asc * sc
        for j, sp in enumerate(sps):
            out = out + alpha_sub[i][j] * sp
    return out

## Part 3: Synthetic Coronagraph Image / 합성 코로나그래프 영상

We build a 256×256 image that mimics a LASCO-style coronagraph: an occulting disk in the middle, a steep radial brightness drop, two streamers, several thin radial filaments (loop-like features), and Poisson-style noise.

256×256 합성 코로나그래프 영상 생성: 중앙 occulter, 가파른 방사형 감쇠, 두 개의 streamer, 몇 개의 얇은 radial filament(loop-like 구조), Poisson 잡음.

In [ ]:
def synthetic_coronagraph(size: int = 256) -> np.ndarray:
    """Build a 2D synthetic coronagraph image with streamers and loops.

    Args:
        size: Image side length.

    Returns:
        2D float array with a steep radial gradient, structures, and noise.
    """
    yy, xx = np.meshgrid(np.linspace(-1, 1, size), np.linspace(-1, 1, size), indexing='ij')
    r = np.sqrt(xx ** 2 + yy ** 2)
    phi = np.arctan2(yy, xx)
    # steep K-corona-like radial drop
    radial = np.exp(-3.0 * r) + 1e-3
    # streamers (broad bumps in azimuth)
    streamer = (
        np.exp(-(((phi - 0.0) ** 2) / 0.05)) + np.exp(-(((phi - np.pi) ** 2) / 0.05))
    )
    # narrow loops/filaments
    filaments = (
        np.exp(-(((phi - 0.6) ** 2) / 0.001))
        + np.exp(-(((phi + 0.4) ** 2) / 0.001))
        + np.exp(-(((phi - 2.0) ** 2) / 0.001))
    )
    structure = 1.0 + 0.6 * streamer + 0.25 * filaments
    img = radial * structure
    # mask central occulter
    img[r < 0.18] = 0.0
    # photon-style noise (variance grows with intensity)
    noise = 0.03 * np.sqrt(np.maximum(img, 1e-6)) * np.random.randn(size, size)
    return img + noise


img = synthetic_coronagraph(256)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(img, cmap='inferno'); axes[0].set_title('Raw'); axes[0].axis('off')
axes[1].imshow(np.log10(img + 1e-3), cmap='inferno')
axes[1].set_title('Raw (log)'); axes[1].axis('off')
plt.tight_layout(); plt.show()

## Part 4: Noise Estimation and Hard-Thresholding / 노이즈 추정과 hard-threshold

Two ingredients are combined for thresholding:
1. **Per-scale stdev factor** $\hat\sigma_j$: simulate unit-variance Gaussian noise and pass it through the same wavelet transform — read off the per-scale stdev. This depends only on the kernel, not on the image.
2. **Local stdev** $\sigma_1(k,l)$: stdev of the finest first-level plane in a 5×5 neighbourhood at every pixel.

The threshold at scale $j$, position $(k,l)$ is $3 \cdot \hat\sigma_j \cdot \sigma_1(k,l)$.

임계는 두 요소의 곱이다: ① per-scale 인자 $\hat\sigma_j$는 단위분산 잡음을 같은 변환에 통과시켜 측정 (영상 무관), ② 국소 stdev $\sigma_1(k,l)$은 영상에서 finest first-level plane의 5×5 이웃 표준편차. 임계는 $3\hat\sigma_j\sigma_1(k,l)$이다.

In [ ]:
def noise_per_scale_factors(n_scales: int, image_size: int = 128, n_runs: int = 3) -> List[float]:
    """Estimate per-scale noise stdev factors for the à trous transform.

    Args:
        n_scales: Number of wavelet planes.
        image_size: Side length of the simulated noise image.
        n_runs: Number of Monte Carlo runs to average.

    Returns:
        List of length n_scales with the empirical stdev at each scale
        when fed unit-variance Gaussian input.
    """
    accum = np.zeros(n_scales)
    for _ in range(n_runs):
        noise_img = np.random.randn(image_size, image_size)
        planes, _ = atrous_decompose(noise_img, n_scales=n_scales)
        accum = accum + np.array([float(p.std()) for p in planes])
    return list(accum / n_runs)


def local_stdev(plane: np.ndarray, window: int = 5) -> np.ndarray:
    """Compute per-pixel local standard deviation in a square window.

    Args:
        plane: 2D array.
        window: Square window side length (odd).

    Returns:
        2D array of the same shape as plane with the local stdev at each pixel.
    """
    kernel = np.ones((window, window)) / float(window * window)
    mu = convolve(plane, kernel, mode='reflect')
    var = convolve((plane - mu) ** 2, kernel, mode='reflect')
    return np.sqrt(np.maximum(var, 1e-12))


def hard_threshold_packet(
    subplanes: List[List[np.ndarray]],
    sigma_factors_first: Sequence[float],
    sigma_factors_sub: Sequence[float],
    sigma_local: np.ndarray,
    k: float = 3.0,
) -> List[List[np.ndarray]]:
    """Apply 3-sigma hard-thresholding to every sub-plane.

    Args:
        subplanes: Output of packet_decompose.
        sigma_factors_first: Per-scale factors for the first-level transform.
        sigma_factors_sub: Per-scale factors for the sub-decomposition.
        sigma_local: 2D local stdev image (same shape as each sub-plane).
        k: Threshold multiple (k=3 → 99.7% confidence).

    Returns:
        Threshold-cleaned sub-planes (same shape).
    """
    cleaned: List[List[np.ndarray]] = []
    for i, sps in enumerate(subplanes):
        cleaned_row: List[np.ndarray] = []
        for j, sp in enumerate(sps):
            sigma_eff = sigma_factors_first[i] * sigma_factors_sub[j] * sigma_local
            sp_clean = np.where(np.abs(sp) >= k * sigma_eff, sp, 0.0)
            cleaned_row.append(sp_clean)
        cleaned.append(cleaned_row)
    return cleaned

## Part 5: Run the Pipeline and Compare Reconstruction Strategies / 파이프라인 실행과 재합성 전략 비교

Run the four-step pipeline with three different weight vectors $\{\alpha_{i,j}\}$ — one mimicking unsharp masking (weight=4 on every detail), one suppressing the continuum, and one boosting only mid-frequency planes. Compare against the raw image.

4단계 파이프라인을 세 가지 가중치 벡터로 실행한다 — (i) unsharp masking 흉내 (모든 detail=4), (ii) continuum 억제, (iii) 중간 주파수 강조. 원본과 비교한다.

In [ ]:
n_first = 4
n_sub = 3
subplanes, sub_continua, top_continuum = packet_decompose(img, n_first=n_first, n_sub=n_sub)

# noise estimation
sf_first = noise_per_scale_factors(n_first, image_size=128, n_runs=3)
sf_sub = noise_per_scale_factors(n_sub, image_size=128, n_runs=3)
first_planes, _ = atrous_decompose(img, n_scales=1)
sigma_local = local_stdev(first_planes[0], window=5)
print('per-scale factors (first):', [round(s, 3) for s in sf_first])
print('per-scale factors (sub):  ', [round(s, 3) for s in sf_sub])

# threshold
subplanes_clean = hard_threshold_packet(subplanes, sf_first, sf_sub, sigma_local, k=3.0)

# Strategy A: unsharp-masking-like (continuum=1, all details=4)
alpha_A_top = 1.0
alpha_A_sub_cont = [4.0] * n_first
alpha_A_sub = [[4.0] * n_sub for _ in range(n_first)]
img_A = packet_reconstruct(
    subplanes_clean, sub_continua, top_continuum,
    alpha_A_top, alpha_A_sub_cont, alpha_A_sub,
)

# Strategy B: drop continuum and emphasise high-frequency (boundaries)
alpha_B_top = 0.0
alpha_B_sub_cont = [0.0] * n_first
alpha_B_sub = [[6.0, 4.0, 2.0] for _ in range(n_first)]
img_B = packet_reconstruct(
    subplanes_clean, sub_continua, top_continuum,
    alpha_B_top, alpha_B_sub_cont, alpha_B_sub,
)

# Strategy C: boost mid-frequency only (loop bodies)
alpha_C_top = 1.0
alpha_C_sub_cont = [1.0, 1.0, 1.0, 1.0]
alpha_C_sub = [[2.0, 6.0, 2.0] for _ in range(n_first)]
img_C = packet_reconstruct(
    subplanes_clean, sub_continua, top_continuum,
    alpha_C_top, alpha_C_sub_cont, alpha_C_sub,
)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(img, cmap='inferno'); axes[0].set_title('Raw'); axes[0].axis('off')
axes[1].imshow(img_A, cmap='inferno'); axes[1].set_title('A: unsharp-style'); axes[1].axis('off')
axes[2].imshow(img_B, cmap='inferno'); axes[2].set_title('B: high-freq only'); axes[2].axis('off')
axes[3].imshow(img_C, cmap='inferno'); axes[3].set_title('C: mid-freq boost'); axes[3].axis('off')
plt.tight_layout(); plt.show()

## Part 6: Quantitative Comparison and Plane Visualisation / 정량 비교와 plane 시각화

Compute simple measures (image-wide stdev, count of pixels above a contrast threshold) and visualise the threshold-cleaned wavelet planes.

간단한 측정량(영상 전체 표준편차, 대비 임계 초과 픽셀 수)을 계산하고, 정제된 wavelet plane들을 시각화한다.

In [ ]:
def contrast_stats(im: np.ndarray, thresh: float = 0.05) -> Tuple[float, int]:
    """Return image stdev and count of pixels with |x - mean| > thresh."""
    sd = float(im.std())
    cnt = int(np.sum(np.abs(im - im.mean()) > thresh))
    return sd, cnt


for name, arr in zip(
    ['Raw', 'A: unsharp-style', 'B: high-freq only', 'C: mid-freq boost'],
    [img, img_A, img_B, img_C],
):
    sd, cnt = contrast_stats(arr, thresh=0.05)
    print(f'{name:25s}  stdev={sd:.3f}  px>0.05={cnt:6d}')

# Plot the cleaned sub-planes for the first first-level scale.
fig, axes = plt.subplots(1, n_sub, figsize=(12, 4))
for j in range(n_sub):
    axes[j].imshow(subplanes_clean[0][j], cmap='gray')
    axes[j].set_title(f'plane (0,{j+1})'); axes[j].axis('off')
plt.tight_layout(); plt.show()

## Summary / 요약

The notebook reproduces the Stenborg-Cobelli pipeline end-to-end on a synthetic 256×256 coronagraph image. The four steps — à trous decomposition, packet refinement, $3\sigma$ local hard-threshold, weighted recomposition — are implemented with NumPy/SciPy only and produce qualitatively distinct outputs depending on the choice of weight vector $\{\alpha_{i,j}\}$.

본 노트북은 256×256 합성 코로나그래프 영상에 Stenborg-Cobelli 파이프라인을 단대단 재현한다. 4단계 — à trous 분해, packet 정제, $3\sigma$ 국소 hard-threshold, 가중 재합성 — 가 NumPy/SciPy만으로 구현되며, 가중치 $\{\alpha_{i,j}\}$에 따라 정성적으로 다른 출력을 만든다.

| Concept / 개념 | This Paper / 이 논문 | Modern Equivalent / 현대 동등물 |
|---|---|---|
| Multiscale decomposition | à trous wavelet (B3-spline) | Pyramidal CNN, Gaussian/Laplacian pyramids |
| Refinement of detail planes | Wavelet packets (recursive split) | Multi-scale transformer encoders |
| Per-pixel noise model | Local stdev × per-scale factor | BatchNorm / GroupNorm at multiple scales |
| Threshold | Hard-threshold $3\sigma$ | Soft-threshold (proximal operator), learned thresholds |
| Weighted recomposition | User-tuned $\alpha_{i, j}$ | Attention-weighted feature pyramid networks (FPN) |